In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("ggplot")

In [ ]:
PROPS = pd.read_csv("../output/props.csv", index_col="run_id")
VARIATION_INDICES = [
    [0, 5], [5, 15], [15, 20]
]
PROPS

In [ ]:
frames = [pd.read_csv("../output/dynamic_results.csv")]
for i in range(2, 1000):
    try:
        frames.append(pd.read_csv(f"../output/dynamic_results_{i}.csv"))
    except FileNotFoundError:
        print(f"Stopping at {i}")
        break
SIM_DF_RAW = pd.concat(frames)

SIM_DFS = [SIM_DF_RAW[(SIM_DF_RAW.run_id >= start) & (SIM_DF_RAW.run_id < end)].groupby("tick").mean() for start, end in VARIATION_INDICES]
SIM_DFS[0]

In [ ]:
PLOT_LIMITS = [
    {"start": 25, "end": 128},
]
PLOT_COLUMNS = [
    {
        "y.2": "total_white",
        "y.3": "total_purple",
        "y.4": "total_yellow",
        "y.5": "total_red",
        "y.6": "total_blue",
        "y.7": "total_orange",
        "y.8": "total_magenta",
        "y.9": "total_green",
    },
]

frames = []

for i in range(1000):
    try:
        inner_frames = []
        for j in range(1):
            df = pd.read_csv(f"../validation/run{i}.csv", skiprows=PLOT_LIMITS[j]["start"], nrows=PLOT_LIMITS[j]["end"] - PLOT_LIMITS[j]["start"] - 1)
            df = df.filter(["x", "y.2", "y.3", "y.4", "y.5", "y.6", "y.7", "y.8", "y.9"], axis=1)
            df = df.iloc[1:]
            df = df.convert_dtypes(convert_integer=True)
            df = df.rename({"x": "tick", **PLOT_COLUMNS[j]}, axis=1)
            df.tick -= 0.5
            df = df.set_index("tick")
            inner_frames.append(df)
        total_frame = pd.concat(inner_frames, axis=1, join="inner")
        total_frame["run_id"] = i
        frames.append(total_frame)
    except FileNotFoundError:
        print(f"Stopped at {i}")
        break

REAL_DF_RAW = pd.concat(frames)
REAL_DFS = [REAL_DF_RAW[(REAL_DF_RAW.run_id >= start) & (REAL_DF_RAW.run_id < end)].groupby("tick").mean() for start, end in VARIATION_INDICES]
REAL_DFS[0]

In [ ]:
for i in range(len(VARIATION_INDICES)):
    plt.figure(figsize=(16, 9))
    plt.plot(SIM_DFS[i].total_blue, color="blue", label="[RepastHPC] Total blue")
    plt.plot(REAL_DFS[i].total_blue, color="blue", label="[NetLogo] Total blue", marker=".", linewidth=0)
    plt.plot(SIM_DFS[i].total_orange, color="orange", label="[RepastHPC] Total orange")
    plt.plot(REAL_DFS[i].total_orange, color="orange", label="[NetLogo] Total orange", marker=".", linewidth=0)
    plt.plot(SIM_DFS[i].total_red, color="red", label="[RepastHPC] Total red")
    plt.plot(REAL_DFS[i].total_red, color="red", label="[NetLogo] Total red", marker=".", linewidth=0)
    plt.plot(SIM_DFS[i].total_yellow, color="yellow", label="[RepastHPC] Total yellow")
    plt.plot(REAL_DFS[i].total_yellow, color="yellow", label="[NetLogo] Total yellow", marker=".", linewidth=0)
    plt.plot(SIM_DFS[i].total_green, color="green", label="[RepastHPC] Total green")
    plt.plot(REAL_DFS[i].total_green, color="green", label="[NetLogo] Total green", marker=".", linewidth=0)
    plt.plot(SIM_DFS[i].total_purple, color="purple", label="[RepastHPC] Total purple")
    plt.plot(REAL_DFS[i].total_purple, color="purple", label="[NetLogo] Total purple", marker=".", linewidth=0)
    plt.plot(SIM_DFS[i].total_magenta, color="magenta", label="[RepastHPC] Total magenta")
    plt.plot(REAL_DFS[i].total_magenta, color="magenta", label="[NetLogo] Total magenta", marker=".", linewidth=0)
    plt.plot(SIM_DFS[i].total_white, color="black", label="[RepastHPC] Total white")
    plt.plot(REAL_DFS[i].total_white, color="black", label="[NetLogo] Total white", marker=".", linewidth=0)
    plt.title('Epistemic share over time')
    plt.xlabel('Tick')
    plt.ylabel('Agent count')
    plt.legend()
    plt.show()